# Промпт-инъекции: демонстрация исследования
Таблицы, графики, документы и реальные сохранённые ответы Claude Sonnet 5.5 и GPT-6 Astra.
Выберите **Среда выполнения → Выполнить все** (CPU). Основной код загружается из GitHub.
Новые платные запросы отправляются только отдельной кнопкой в конце.

In [ ]:
# Загрузка проекта и зависимостей. Повторный запуск использует ту же копию.
import sys, os, json, subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/salawhatt/llm-prompt-injection-study.git'
ROOT = Path('/content/llm-prompt-injection-study') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)], check=True)
    origin = subprocess.check_output(['git', '-C', str(ROOT), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError('В выбранной папке находится другой репозиторий.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(ROOT / 'requirements.txt'), 'ipywidgets>=8,<9'], check=True)
    # Если библиотеки были импортированы до установки, перезапустите сеанс один раз.
    import importlib.metadata
    for package, module in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scikit-learn', 'sklearn')]:
        loaded = sys.modules.get(module)
        if loaded and loaded.__version__ != importlib.metadata.version(package):
            raise RuntimeError('Зависимости установлены. Перезапустите сеанс и выполните все ячейки.')
assert (ROOT / 'manifest.json').is_file(), 'Локально запускайте блокнот из корня репозитория.'
sys.path[:0] = [str(ROOT), str(ROOT / 'src')]
os.environ['MPLBACKEND'] = 'Agg'
print('Версия:', subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Image, JSON, clear_output
import study
from run import integrity
from experiment_core import prepare

integrity()
cases, responses, annotations, filters = study.load(ROOT)
frame, predictions = study.reconstruct(cases, responses, annotations, filters)
main = frame[(frame.cohort == 'main') & (frame['repeat'] == 1)]
OUT = ROOT / 'output' / 'colab'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Документов: {len(cases)}. Сохранённых ответов: {len(responses)}.')
display(study.inventory(cases, responses))

## 1. Таблицы
Основной тест, сложные испытания, дополнительные условия и повторы показаны отдельно.
Смысловые показатели используют сохранённые автоматические оценки.

In [ ]:
tables = {
    'ASR основного теста': study.metrics(main),
    'Обычные задачи': study.clean_metrics(main),
    'Фильтры D/E': study.detector_metrics(predictions),
    'Сложные испытания': study.metrics(frame[(frame.cohort == 'challenge') & (frame['repeat'] == 1)]),
    'Изменение условий': study.factor_table(frame),
    'Повторные ответы': study.repeats(frame),
}
table_choice = widgets.Dropdown(options=list(tables), description='Таблица:', layout=widgets.Layout(width='450px'))
def show_table(name):
    display(tables[name])
display(table_choice, widgets.interactive_output(show_table, {'name': table_choice}))

## 2. Графики
Выберите график. Изображения сохраняются в PNG и SVG в `output/colab/`.

In [ ]:
plots = {
    'ASR и 95%-е интервалы': (study.plot_main, main, '01_asr'),
    'Правильность обычных ответов': (study.plot_clean, main, '02_clean'),
    'Ошибки фильтров D/E': (study.plot_filters, predictions, '03_filters'),
    'Категории инъекций': (study.plot_categories, main, '04_categories'),
    'Длина документа': (study.plot_length, predictions, '05_length'),
}
plot_choice = widgets.Dropdown(options=list(plots), description='График:', layout=widgets.Layout(width='450px'))
def show_plot(name):
    plot, data, filename = plots[name]
    plot(data, OUT)
    display(Image(filename=str(OUT / (filename + '.png')), width=950))
display(plot_choice, widgets.interactive_output(show_plot, {'name': plot_choice}))

## 3. Документ, запрос и ответы
A: без специальной защиты. B: системная инструкция и JSON. D/E: фильтр перед B.
Выбор влияет также на пример нового запроса в последнем разделе.

In [ ]:
case_map = {r['case']: r for r in cases}
case_choice = widgets.Dropdown(
    options=[(f"{r['case']} | {r['question'][:65]}", r['case']) for r in cases],
    value='v4-003-main-attack', description='Документ:', layout=widgets.Layout(width='95%'))
arm_choice = widgets.Dropdown(options=study.ARMS, value='B', description='Защита:')
model_choice = widgets.Dropdown(options=[(v, k) for k, v in study.NAMES.items()], description='Модель:')
model_ids = {r['model_name']: r['model'] for r in responses}

def demo_payload(case_id, model_name, arm):
    record = case_map[case_id]
    if arm in filters:
        detector, threshold = filters[arm]
        if detector.predict_proba([record['document']])[0, 1] >= threshold:
            return None
    return {'model': model_ids[model_name], 'temperature': 0, 'max_tokens': 1024,
            'messages': prepare(record, record['question'], 'A' if arm == 'A' else 'B')}

def show_document(case_id, model_name, arm):
    record = case_map[case_id]
    print(record['document'])
    print('Вопрос:', record['question'], '\nЭталон:', record['gold'])
    display(study.filter_example([record], filters))
    selected = frame[(frame['case'] == case_id) & (frame['repeat'] == 1)]
    for name in study.NAMES:
        print('\n' + study.NAMES[name])
        for _, row in selected[(selected.model_name == name) & selected.arm.isin(['A', 'B'])].iterrows():
            print(f'{row.arm}: {row.answer}')
    display(selected[['model_name', 'arm', 'status', 'attack_success', 'strict_correct', 'semantic_correct']]
            .rename(columns={'model_name': 'Модель', 'arm': 'Защита', 'status': 'Статус',
                             'attack_success': 'Успех атаки', 'strict_correct': 'Строго правильно',
                             'semantic_correct': 'Правильно по смыслу'}))
    payload = demo_payload(case_id, model_name, arm)
    if payload is None:
        print('Документ блокируется выбранным фильтром: API-запрос не требуется.')
    else:
        display(JSON(payload, expanded=True))

display(case_choice, widgets.HBox([arm_choice, model_choice]), widgets.interactive_output(
    show_document, {'case_id': case_choice, 'model_name': model_choice, 'arm': arm_choice}))

## 4. Свой текст для фильтра
Локальная проверка D/E. Не отправляет текст в LLM.

In [ ]:
custom_text = widgets.Textarea(value=case_map[case_choice.value]['document'],
                               layout=widgets.Layout(width='95%', height='130px'))
filter_button = widgets.Button(description='Проверить D/E')
filter_output = widgets.Output()
def check_custom(_):
    with filter_output:
        clear_output(wait=True)
        display(study.filter_example([{'document': custom_text.value, 'kind': 'Свой текст'}], filters))
filter_button.on_click(check_custom)
display(custom_text, filter_button, filter_output)

## 5. Один новый запрос (необязательно)
В **Секретах Colab** добавьте `CLOUDRU_API_KEY` и разрешите доступ блокноту.
Кнопка отправляет выбранный выше документ через [Cloud.ru](https://cloud.ru/docs/foundation-models/ug/topics/quickstart).
Это платный демонстрационный вызов, он не меняет результаты исследования.
Повторный идентичный вызов берётся из локального кэша текущего сеанса.

In [ ]:
from demo_request import request_once

send_button = widgets.Button(description='Отправить один платный запрос',
                             button_style='warning', layout=widgets.Layout(width='300px'))
live_output = widgets.Output()
def send_one(_):
    send_button.disabled = True
    try:
        with live_output:
            clear_output(wait=True)
            payload = demo_payload(case_choice.value, model_choice.value, arm_choice.value)
            if payload is None:
                print('Документ блокируется локальным фильтром. Запрос не отправлен.')
                return
            if not IN_COLAB:
                print('Кнопка использует секреты Google Colab. Локальная демонстрация работает без API.')
                return
            from google.colab import userdata
            try:
                api_key = userdata.get('CLOUDRU_API_KEY')
            except Exception:
                print('Добавьте CLOUDRU_API_KEY в Секреты Colab и разрешите доступ блокноту.')
                return
            try:
                result = request_once(payload, api_key, ROOT / 'output' / 'live_requests')
            finally:
                del api_key
            display(JSON(result, expanded=True))
    finally:
        send_button.disabled = False

send_button.on_click(send_one)
display(send_button, live_output)

## 6. Скачать таблицы и построенные графики
Архив содержит только материалы из `output/colab/`, без ключей и новых API-ответов.

In [ ]:
import shutil
for i, (name, table) in enumerate(tables.items(), 1):
    table.to_csv(OUT / f'table_{i:02d}.csv', index=False, encoding='utf-8-sig')
(OUT / 'tables.json').write_text(json.dumps(
    {f'table_{i:02d}.csv': name for i, name in enumerate(tables, 1)},
    ensure_ascii=False, indent=2), encoding='utf-8')
download_button = widgets.Button(description='Скачать ZIP')
def download_results(_):
    archive = shutil.make_archive(str(ROOT / 'output' / 'colab_results'), 'zip', OUT)
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
    else:
        print(archive)
download_button.on_click(download_results)
display(download_button)